# IMPORTANT NOTE: Widget Functionality Removed

This notebook has been modified to remove its dependency on `ipywidgets`. All interactive widget-based inputs have been replaced with direct variable assignments within the code cells.

**Please look for comments marked `# USER ACTION:` throughout the notebook.** You will need to manually set the values for these variables according to your project's needs before executing the cells.

This change allows the notebook to be run in environments where `ipywidgets` may not be fully supported or where non-interactive execution is preferred.

<img align="left" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="right">Upload clips or frames to Zooniverse</h1>
<h3 align="right"><a href="https://colab.research.google.com/github/ocean-data-factory-sweden/kso/blob/main/notebooks/classify/Upload_subjects_to_Zooniverse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a></h3>
<h3 align="right">Written by the KSO Team</h3>

This notebook takes you through the process of:

* Connecting to a Zooniverse project

* Extracting clips from videos stored locally or in the cloud.

* Modify these clips with for example a color correction or video compression.

* Upload the clips to Zooniverse for the 1st part of the species identification workflow (when does X species appear on a video).

If you do not have a project with us yet, you can run the template project to get a taste of how it all works. Only the uploading of the clips to Zooniverse will not be possible if you are not a member of our template project on Zooniverse.

🔴 <span style="color:red">&nbsp;NOTE: If you want to run another project than the template project, you need to have a Zooniverse account and be a member of the corresponding project.  </span>

# Set up KSO requirements

### Install requirements and load KSO modules

Installing the requirements in Google Colab takes ~4 mins and might automatically crash/restart the session. Please run this cell until you get the "KSO successfully imported!" message.

In [ ]:
%matplotlib inline
import os
import sys


def initiate_dev_version():
    kso_path = os.path.abspath(os.path.join(os.getcwd(), "../.."))
    if os.path.isdir(os.path.join(kso_path, "kso_utils")):
        sys.path.insert(0, kso_path)
        %load_ext autoreload
        %autoreload 2
        print("Development mode ON - kso-utils added to the system.")
    else:
        raise FileNotFoundError("kso_utils directory not found in the expected path.")


def install_kso_utils():
    !pip install -q kso-utils
    # Temporary workaround to install panoptes from the source (avoid requests incompatibility)
    !pip install git+https://github.com/zooniverse/panoptes-python-client.git
    print("Restarting runtime to apply package changes...")
    os.kill(os.getpid(), 9)


try:
    initiate_dev_version()
    import kso_utils.widgets as kso_widgets
    import kso_utils.project_utils as p_utils
    from kso_utils.project import ProjectProcessor

    print("KSO dev successfully imported!")
except Exception as e:
    install_kso_utils()
    import kso_utils.widgets as kso_widgets
    import kso_utils.project_utils as p_utils
    from kso_utils.project import ProjectProcessor

    print("KSO PyPi successfully imported!")

### Specify GPU availability

In [ ]:
# gpu_available = kso_widgets.gpu_select()
gpu_available_result = False # USER ACTION: Set to True if GPU is available and desired, False otherwise.


### Choose your project

In [ ]:
# project_name = kso_widgets.choose_project()
project_name_value = "template" # USER ACTION: Set this to your Zooniverse project name string (e.g., from projects_list.csv).


### Initiate project's database

In [ ]:
# Save the name of the project
project = p_utils.find_project(project_name=project_name_value)

# Initiate pp
pp = ProjectProcessor(project)

### Specify to request (or not) the latest Zooniverse info

In [ ]:
# latest_zoo_info = kso_widgets.request_latest_zoo_info()
latest_zoo_info_result = False # USER ACTION: Set to True to request latest Zooniverse info, False to use last available.


### Connect and retrieve information from the Zooniverse project

In [ ]:
pp.connect_zoo_project(latest_zoo_info_result)

# Task 1: Upload clips to Zooniverse

### Specify movie of interest

In [ ]:
# USER ACTION: Specify the movie you want to process.
# This was previously selected via a widget.
# You need to provide the movie filename.
movie_filename_to_process = "your_movie_filename.mp4" # USER ACTION: Replace with your actual movie filename

# The ProjectProcessor object (pp) needs its internal state updated as if a movie was selected.
# This typically involves setting attributes like 'pp.selected_movies_df', 'pp.selected_movie_path',
# 'pp.selected_movies', and 'pp.selected_movies_ids'.

if hasattr(pp, 'available_movies_df') and not pp.available_movies_df.empty:
    selected_movie_info_df = pp.available_movies_df[pp.available_movies_df['filename'] == movie_filename_to_process]
    if not selected_movie_info_df.empty:
        pp.selected_movies_df = selected_movie_info_df.copy()
        # Determine movie path. This might need adjustment based on your project structure
        # and how 'full_path' or similar is stored in 'available_movies_df'.
        if 'full_path' in pp.selected_movies_df.columns:
            pp.selected_movie_path = pp.selected_movies_df['full_path'].tolist()
        elif hasattr(pp.project, 'movie_folder') and pp.project.movie_folder:
            import os
            pp.selected_movie_path = [os.path.join(pp.project.movie_folder, fname) for fname in pp.selected_movies_df['filename']]
        else:
            pp.selected_movie_path = pp.selected_movies_df['filename'].tolist() # Fallback, may not be full path
            print(f"Warning: Could not determine full path for {movie_filename_to_process}. Using filename as path.")

        pp.selected_movies = pp.selected_movies_df['filename'].tolist()
        if 'movie_id' in pp.selected_movies_df.columns:
            pp.selected_movies_ids = pp.selected_movies_df['movie_id'].tolist()
        else:
            # If movie_id is not in available_movies_df, you might need to fetch or assign it.
            # For this example, we'll create a placeholder if it's missing.
            print(f"Warning: 'movie_id' not found for {movie_filename_to_process}. Using placeholder.")
            pp.selected_movies_ids = [f"id_{fname}" for fname in pp.selected_movies]

        print(f"Movie '{movie_filename_to_process}' (ID(s): {pp.selected_movies_ids}) selected for processing.")
        print(f"Selected movie DataFrame (pp.selected_movies_df):\n{pp.selected_movies_df}")
        print(f"Selected movie path(s) (pp.selected_movie_path): {pp.selected_movie_path}")
    else:
        print(f"ERROR: Movie '{movie_filename_to_process}' not found in pp.available_movies_df. Please check filename and project setup.")
        print(f"Available movies are: {pp.available_movies_df['filename'].tolist() if hasattr(pp, 'available_movies_df') else 'Not available'}")
        # As a fallback, create a minimal pp.selected_movies_df
        # import pandas as pd
        # pp.selected_movies_df = pd.DataFrame({'filename': [movie_filename_to_process], 'movie_id': [f"id_{movie_filename_to_process}"]}) # Add other essential columns if known
        # pp.selected_movie_path = [movie_filename_to_process] # User must ensure this path is correct
        # pp.selected_movies = [movie_filename_to_process]
        # pp.selected_movies_ids = [f"id_{movie_filename_to_process}"]
        # print("Created a fallback pp.selected_movies_df. Ensure 'movie_id' and other details are correct.")

else:
    print("ERROR: pp.available_movies_df is not available or empty. Cannot automatically select movie.")
    print("USER ACTION: You may need to manually create pp.selected_movies_df and related attributes (see commented example in this cell).")

# Ensure the attributes are set for the next steps
if not hasattr(pp, 'selected_movies_df') or pp.selected_movies_df.empty:
        raise Exception("pp.selected_movies_df was not set. Please check movie selection cell.")

In [ ]:
# pp.choose_footage(preview_media=True)
print("Movie selection is now handled in the preceding cell. Ensure it's configured correctly.")


### Check if movie is already in Zooniverse

In [ ]:
# Remember to query the newest zooniverse data to get the most up to date list of clips uploaded
pp.check_movies_uploaded_zoo()

## Create some clip examples (Optional)
Test different parameters (e.g. compression rate, color modifications) in randomly selected clip examples

In [ ]:
pp.generate_zoo_clips(
    is_example=True,
    use_gpu=gpu_available_result,
)

In [ ]:
#if code above doesn't work, run this to ensure ffmpeg works as expected
# !conda install -c conda-forge ffmpeg -y -v
# import os
# import subprocess
# import logging
# import tempfile
# from pathlib import Path
# import urllib.request

# def test_ffmpeg_clip_extraction(
#     input_video_url='https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4'
# ):
#     """
#     Test FFmpeg functionality by downloading a standard test video and extracting a clip
#     """
#     # Set up logging
#     logging.basicConfig(level=logging.INFO)
#     logger = logging.getLogger(__name__)
    
#     try:
#         # Create temporary directories
#         with tempfile.TemporaryDirectory() as temp_dir:
#             # Paths
#             input_video_path = Path(temp_dir) / 'input_video.mp4'
#             output_clip_path = Path(temp_dir) / 'output_clip.mp4'
            
#             # Download test video using urllib
#             logger.info(f"Downloading video from {input_video_url}")
#             urllib.request.urlretrieve(input_video_url, input_video_path)
            
#             # Verify download
#             if not input_video_path.exists():
#                 logger.error("Failed to download video")
#                 return False
            
#             # FFmpeg clip extraction
#             ffmpeg_cmd = [
#                 'ffmpeg', 
#                 '-i', str(input_video_path),  # Input file
#                 '-t', '10',                   # Duration of clip (10 seconds)
#                 '-c', 'copy',                 # Stream copy mode (fast)
#                 str(output_clip_path)         # Output file
#             ]
            
#             # Run FFmpeg
#             try:
#                 result = subprocess.run(
#                     ffmpeg_cmd, 
#                     capture_output=True, 
#                     text=True, 
#                     check=True  # This will raise an exception if the command fails
#                 )
                
#                 # Check output clip exists and has size
#                 if output_clip_path.exists() and output_clip_path.stat().st_size > 0:
#                     logger.info("✅ FFmpeg clip extraction successful!")
#                     logger.info(f"Output clip size: {output_clip_path.stat().st_size} bytes")
#                     return True
#                 else:
#                     logger.error("❌ FFmpeg clip extraction failed")
#                     return False
            
#             except subprocess.CalledProcessError as e:
#                 logger.error(f"FFmpeg Command Error: {e}")
#                 logger.error(f"stdout: {e.stdout}")
#                 logger.error(f"stderr: {e.stderr}")
#                 return False
    
#     except Exception as e:
#         logger.error(f"Unexpected error: {e}")
#         return False

# def check_ffmpeg_version():
#     """
#     Check FFmpeg installation and version
#     """
#     try:
#         result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
#         print("FFmpeg Version:")
#         print(result.stdout.split('\n')[0])
#         return True
#     except FileNotFoundError:
#         print("FFmpeg is not installed or not in PATH")
#         return False

# # Run the checks (modify for notebook)
# print("Checking FFmpeg Version:")
# check_ffmpeg_version()

# print("\nChecking FFmpeg Clip Extraction:")
# test_ffmpeg_clip_extraction()

In [ ]:
pp.check_clip_size()

In [ ]:
# kso_widgets.compare_clips(
#     example_clips=pp.generated_clips.clip_path,
#     modified_clips=pp.generated_clips.clip_example_original_path,
# )
print("Clip comparison widget removed. Visual comparison of clips is skipped.")


## Create the clips to upload to Zooniverse

In [ ]:
pp.generate_zoo_clips(
    is_example=False,
    use_gpu=gpu_available_result,
)

### Check the size of the clips

In [ ]:
pp.check_clip_size()

## Upload clips to Zooniverse

If you receive an error message related to file size, we recommend shortening the clip length or compressing the clips to achieve a suitable filesize.

Make sure your workflows in Zooniverse have different names to avoid issues while creating a new workflow

🔴 <span style="color:red">&nbsp;NOTE: If you run the template project without being a member of our template project, it is not possible to run this last cell.  </span>

In [ ]:
pp.upload_zoo_subjects("clip")

In [ ]:
# END OF TASK 1

# Task 2: Upload images (frames) to Zooniverse

## Select Zooniverse workflow id and version of interest

##### Note: Make sure your workflows in Zooniverse have different names to avoid issues while selecting the workflow id

### Choose the workflows and versions of interest

In [ ]:
# USER ACTION: Manually specify Zooniverse workflow details.
# This was previously done using an interactive widget.
# We need to mock the structure that pp.choose_zoo_workflows() would have created,
# particularly pp.workflow_widget.checks, if subsequent code relies on it.

class MockWorkflowWidget: # Basic mock class
    def __init__(self):
        self.checks = {}
        self.value = {} # if .value is accessed

pp.workflow_widget = MockWorkflowWidget()

# USER ACTION: Set these values for your project
workflow_name_value = "Your Workflow Name" # e.g., "Koster Seafloor Observer"
subject_type_value = "clip"  # "clip" or "frame"
min_workflow_version_value = "1.0" # Minimum version of the workflow to consider

pp.workflow_widget.checks["Workflow name: #0"] = workflow_name_value
pp.workflow_widget.checks["Subject type: #0"] = subject_type_value
pp.workflow_widget.checks["Minimum workflow version: #0"] = min_workflow_version_value

# If the choose_zoo_workflows also sets other attributes on pp directly, those would need to be set here too.
# For example, it might set pp.selected_workflow_id.
# This requires knowledge of the specific implementation of choose_zoo_workflows.
# Assuming for now that pp.workflow_widget.checks is the main output needed.
# If pp.connect_zoo_project() populates pp.workflows_df, we can try to get an ID:
if hasattr(pp, 'workflows_df') and hasattr(pp.workflows_df, 'empty') and not pp.workflows_df.empty:
    workflow_details = pp.workflows_df[pp.workflows_df['display_name'] == workflow_name_value]
    if not workflow_details.empty:
        # Potentially select the one matching min_workflow_version_value or latest
        pp.selected_workflow_id = workflow_details['workflow_id'].iloc[0] # Example: take first match
        print(f"Selected workflow ID based on name: {pp.selected_workflow_id}")
    else:
        print(f"Warning: Workflow named '{workflow_name_value}' not found in pp.workflows_df.")
        pp.selected_workflow_id = "00000" # Placeholder ID
else:
    print("Warning: pp.workflows_df not available or empty to derive workflow ID. Using placeholder.")
    pp.selected_workflow_id = "00000" # Placeholder ID


print(f"Workflow details set manually: Name='{pp.workflow_widget.checks['Workflow name: #0']}', Type='{pp.workflow_widget.checks['Subject type: #0']}', MinVersion='{pp.workflow_widget.checks['Minimum workflow version: #0']}'")
print("Ensure these values are correct for your Zooniverse project and that pp.selected_workflow_id is appropriate.")

In [ ]:
# pp.choose_zoo_workflows()
print("Zooniverse workflow selection is now handled in the preceding cell.")


### Sample and process Zooniverse classifications from the workflows of interest

In [ ]:
pp.process_zoo_classifications()

## Aggregate classifications received from the workflows of interest

### Aggregate classifications based on threshold

In [ ]:
# users = kso_widgets.choose_aggregation_users(pp.processed_zoo_classifications)
users_result = [] # USER ACTION: Set to a list of Zooniverse user names (strings) to filter by, or keep as empty list [] for all users.


### Specify the aggregation parameters

In [ ]:
# USER ACTION REQUIRED: Define the subject type for aggregation.
# This should match what was set in the manual workflow setup cell.
if hasattr(pp, 'workflow_widget') and "Subject type: #0" in pp.workflow_widget.checks:
    subject_type_for_aggregation = pp.workflow_widget.checks["Subject type: #0"]
else:
    subject_type_for_aggregation = "clip" # USER ACTION: Fallback, set "clip" or "frame" if above failed.
    print(f"Warning: Could not determine subject_type_for_aggregation from pp.workflow_widget. Defaulting to '{subject_type_for_aggregation}'. Please verify.")
print(f"Using subject_type: {subject_type_for_aggregation} for aggregation parameters.")

# USER ACTION REQUIRED: Set aggregation parameters manually.
# These were previously set using interactive sliders.
# Consult kso_utils/widgets.py `choose_agg_parameters` for context on these values.
agg_tresh_value = 0.8  # Aggregation threshold (proportion)
min_users_value = 3    # Minimum number of users

# Parameters specific to "frame" subject type
object_tresh_value = 0.5 # Object threshold (proportion)
iou_epsilon_value = 0.7  # IOU Epsilon for clustering
inter_user_agreement_value = 0.4 # Inter-user agreement within a cluster

print(f"USER ACTION: Review and set aggregation parameters below if defaults are not suitable.")
print(f"agg_tresh_value = {agg_tresh_value}")
print(f"min_users_value = {min_users_value}")

# This dictionary will be passed to pp.aggregate_zoo_classifications
# The keys should ideally match what the function expects, or be generic if the function processes them based on order/type.
aggregation_parameters_input = {
    'agg_tresh_slider_value': agg_tresh_value, 
    'min_users_slider_value': min_users_value,
}

if subject_type_for_aggregation == "frame":
    print(f"object_tresh_value = {object_tresh_value}")
    print(f"iou_epsilon_value = {iou_epsilon_value}")
    print(f"inter_user_agreement_value = {inter_user_agreement_value}")
    aggregation_parameters_input['agg_obj_slider_value'] = object_tresh_value
    aggregation_parameters_input['agg_iou_slider_value'] = iou_epsilon_value
    aggregation_parameters_input['agg_iua_slider_value'] = inter_user_agreement_value
        
print("Aggregation parameters defined. These will be used in the next step (Cell ID bc4291ce).")

### Aggregate classifications based on parameters

In [ ]:
pp.aggregate_zoo_classifications(aggregation_parameters_input, users_result)


🔴 <span style="color:red">&nbsp;NOTE: If the output from the cell above says that 0 classifications are aggregated, you can experiment with other agreement thresholds, or you need to wait for more annotations to be made in Zooniverse.   </span>

### Select species of interest

In [ ]:
pp.species_of_interest = kso_widgets.choose_species(
    pp.db_connection, pp.aggregated_zoo_classifications["label"].unique()
)

### Extract frames from videos that have species of interest (based on selected aggreement)

In [ ]:
# Get all available frames for the selected species from clips
pp.extract_zoo_frames()

In [ ]:
# Review the size of the frames
pp.check_frame_size()

## Get frames from movies (Manual)

In [ ]:
# Choose folder containing movies
input_folder = kso_widgets.choose_folder()

In [ ]:
# Choose output folder for frames
output_folder = kso_widgets.choose_folder()

In [ ]:
# Generate suitable frames for upload by modifying initial frames
pp.generate_custom_frames(
    input_path=input_folder.selected,
    output_path=output_folder.selected,
    skip_start=120,
    skip_end=120,
    num_frames=10,
    frames_skip=None,
)

### Modify the frames if needed

In [ ]:
pp.modify_zoo_frames()

In [ ]:
# Review the size of the modified clips
pp.check_frame_size()

### Preview the frames

In [ ]:
# Compare the original and modified clips
pp.compare_frames(df=pp.modified_frames)

## Upload frames to Zooniverse

Make sure your workflows in Zooniverse have different names to avoid issues while creating a new workflow

🔴 <span style="color:red">&nbsp;NOTE: If you run the template project without being a member of our template project, it is not possible to run this last cell.  </span>

In [ ]:
pp.upload_zoo_subjects("frame")

In [ ]:
# END OF TASK 2